# Early Prediction of Mortality in Acute Kidney Injury Requiring Dialysis

Reproducible execution pipeline for the study "Early Prediction of Mortality in Acute Kidney Injury Requiring Dialysis: A Comparative Analysis of Tabular Foundation Models and Conventional Machine Learning".

## Overview
This notebook implements the end-to-end modeling and evaluation workflow restricted to predictors available at Time Zero (the calendar day of kidney replacement therapy indication):
1. **Cohort Ingestion & Feature Curation:** Deduplication to index ICU admission, adult cohort filtering, physiological plausibility cleaning, 24-hour kinetic feature engineering, and stratified 80/20 development/holdout test partitioning (`SEED = 42`).
2. **Model Training & Inference:**
   - **Conventional ML:** Logistic Regression, Random Forest, XGBoost, LightGBM, and CatBoost (Optuna 5-fold CV hyperparameter optimization and out-of-fold isotonic calibration across `Raw`, `Minimal`, and `Preprocessed` regimes).
   - **Tabular Foundation Models (Zero-Shot):** Local in-context inference using `TabPFN 3.5` and `TabFM 1.0` (`google/tabfm-1.0.0-pytorch`).
   - **Neural & Language Comparators:** Supervised `FT-Transformer` trained from scratch and fine-tuned `ModernBERT-base` on serialized clinical records.
3. **Evaluation, Fairness & Explainability:** Bootstrap CIs, paired bootstrap tests, calibration diagnostics, Decision Curve Analysis (DCA), Fairlearn Equalized Odds Difference (EOD) audits, and SHAP attributions.

## Required API Keys & Environment Setup
To download pretrained model weights locally, you must provide valid authentication tokens (via Colab Secrets or environment variables; do not hardcode credentials before sharing):
* **`TABPFN_TOKEN` / `TABPFN_API_KEY`:** Required by the `tabpfn` package to download `TabPFN 3.5` weights.
* **`HF_TOKEN`:** Hugging Face access token used to download `google/tabfm-1.0.0-pytorch` and `answerdotai/ModernBERT-base` weights.

All model inference runs locally on the attached GPU without transmitting patient records to external commercial APIs.

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

SEED = 42
INPUT_FILE = "/content/drive/MyDrive/FELIPE/franco_nefro_II/classificacao_obito_desfechos_revisados.xlsx"
OUTPUT_DIR = "/content/drive/MyDrive/FELIPE/franco_nefro_II/output_v7"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load raw clinical registry and preserve adjudication index prior to filtering
df_raw = pd.read_excel(INPUT_FILE)
df_raw['was_adjudicated'] = (df_raw.index < 202).astype(int)

df = df_raw.copy()
df['RGHC'] = df['RGHC'].astype(str).str.strip().str.upper()

# Retain confirmed clinical endpoints (0 = Survival, 1 = In-hospital mortality)
df = df[df['alvo'].isin([0, 1])].copy()
df['alvo'] = df['alvo'].astype(int)

# Deduplicate to index admission (Time Zero)
date_candidates = ["Dt_Iniciou_dialise", "dt_conduta", "Dt_Iniciou"]
avail_date = [c for c in date_candidates if c in df.columns]
if avail_date:
    df['temp_date'] = pd.to_datetime(df[avail_date[0]], errors='coerce')
    df = df.sort_values(by=['RGHC', 'temp_date'], ascending=[True, True], na_position='last')
    df = df.drop_duplicates(subset=['RGHC'], keep='first').drop(columns=['temp_date'])
else:
    df = df.drop_duplicates(subset=['RGHC'], keep='first')

# Clinical parsing utilities
def parse_clinical_numeric(value):
    if pd.isna(value):
        return np.nan
    val_str = str(value).strip()
    if val_str in ["", ".", "-", "--", "NA", "N/A", "NULL", "null", "?", "??", "Sem dado"]:
        return np.nan
    if re.search(r"[a-zA-Z]", val_str) and val_str.upper() not in ["NAN", "INF"]:
        return np.nan
    cleaned = re.sub(r"[^\d,\.-]", "", val_str)
    if cleaned in ["", ".", "-"]:
        return np.nan
    try:
        return float(cleaned.replace(",", "."))
    except ValueError:
        return np.nan

def map_binary_indicator(value):
    if pd.isna(value):
        return 0
    return 1 if str(value).strip().upper() in ["S", "SIM", "TRUE", "1", "1.0"] else 0

# Parse continuous physiological and laboratory predictors
continuous_raw_cols = [
    'id_atual', 'Creat_basal',
    'Creat_anterior_inic_dialise', 'Creat_inic_dialise',
    'Ureia_anterior_inic_dialise', 'Ureia_inic_dialise',
    'PH_anterior_inic_dialise', 'PH_inic_dialise',
    'BIC_anterior_inic_dialise', 'BIC_inic_dialise',
    'Sodio_anterior_inic_dialise', 'Sodio_inic_dialise',
    'Potassio_anterior_inic_dialise', 'Potassio_inic_dialise'
]

for col in continuous_raw_cols:
    if col in df.columns:
        parsed = df[col].apply(parse_clinical_numeric)
        # Convert physiologically impossible zero or negative laboratory/age values to NaN
        df[col] = parsed.where(parsed > 0, np.nan)

# Enforce adult cohort inclusion criterion (age >= 18 years; retains missing ages for imputation)
n_before_age = len(df)
df = df[~(df['id_atual'] < 18)].copy()
n_pediatric_excluded = n_before_age - len(df)

# Parse biological sex and binary clinical flags
if 'sx' in df.columns:
    df['sx'] = df['sx'].map({'M': 0, 'F': 1, '0': 0, '1': 1, 0: 0, 1: 1}).fillna(0).astype(int)

binary_candidates = [c for c in df.columns if c.startswith("mot_convocacao_") or c.startswith("ant_pessoais_")]
for col in binary_candidates:
    df[col] = df[col].apply(map_binary_indicator)

# Engineer 24-hour physiological kinetics and nitrogenous waste ratios safely
df["delta_creat_24h"] = df["Creat_inic_dialise"] - df["Creat_anterior_inic_dialise"]
df["delta_ureia_24h"] = df["Ureia_inic_dialise"] - df["Ureia_anterior_inic_dialise"]
df["delta_ph_24h"] = df["PH_inic_dialise"] - df["PH_anterior_inic_dialise"]
df["delta_bic_24h"] = df["BIC_inic_dialise"] - df["BIC_anterior_inic_dialise"]
df["delta_sodio_24h"] = df["Sodio_inic_dialise"] - df["Sodio_anterior_inic_dialise"]
df["delta_potassio_24h"] = df["Potassio_inic_dialise"] - df["Potassio_anterior_inic_dialise"]

df["ratio_ureia_creat_inic"] = df["Ureia_inic_dialise"] / df["Creat_inic_dialise"]
df["ratio_ureia_creat_basal"] = df["Ureia_inic_dialise"] / df["Creat_basal"]

# Standardized international feature mapping
translation_mapping = {
    'RGHC': 'patient_id',
    'alvo': 'inhospital_mortality',
    'id_atual': 'age_years',
    'sx': 'sex_female',
    'Creat_basal': 'baseline_serum_creatinine',
    'Creat_anterior_inic_dialise': 'predialysis_creatinine_d_minus_1',
    'Creat_inic_dialise': 'dialysis_day_creatinine',
    'Ureia_anterior_inic_dialise': 'predialysis_urea_d_minus_1',
    'Ureia_inic_dialise': 'dialysis_day_urea',
    'PH_anterior_inic_dialise': 'predialysis_ph_d_minus_1',
    'PH_inic_dialise': 'arterial_ph_d0',
    'BIC_anterior_inic_dialise': 'predialysis_bicarbonate_d_minus_1',
    'BIC_inic_dialise': 'serum_bicarbonate_d0',
    'Sodio_anterior_inic_dialise': 'predialysis_sodium_d_minus_1',
    'Sodio_inic_dialise': 'serum_sodium_d0',
    'Potassio_anterior_inic_dialise': 'predialysis_potassium_d_minus_1',
    'Potassio_inic_dialise': 'serum_potassium_d0',
    'delta_creat_24h': 'delta_creatinine_24h',
    'delta_ureia_24h': 'delta_urea_24h',
    'delta_ph_24h': 'delta_ph_24h',
    'delta_bic_24h': 'delta_bicarbonate_24h',
    'delta_sodio_24h': 'delta_sodium_24h',
    'delta_potassio_24h': 'delta_potassium_24h',
    'ratio_ureia_creat_inic': 'urea_to_creatinine_ratio_d0',
    'ratio_ureia_creat_basal': 'urea_to_baseline_creatinine_ratio',
    'mot_convocacao_IRA_I': 'indication_aki_stage_1',
    'mot_convocacao_IRA_II': 'indication_aki_stage_2',
    'mot_convocacao_IRA_III': 'indication_aki_stage_3',
    'mot_convocacao_sepse': 'indication_sepsis',
    'mot_convocacao_hipovolemia': 'indication_hypovolemia',
    'mot_convocacao_choque': 'indication_shock',
    'mot_convocacao_SHR': 'indication_hepatorenal_syndrome',
    'mot_convocacao_pos_cirurgia': 'indication_post_operative',
    'mot_convocacao_GNRP': 'indication_rpgn',
    'mot_convocacao_obstrucao': 'indication_obstruction',
    'mot_convocacao_D_renal_cron': 'indication_chronic_kidney_disease',
    'mot_convocacao_dist_hidroel': 'indication_electrolyte_disorder',
    'mot_convocacao_nefrotoxicidade': 'indication_nephrotoxicity',
    'mot_convocacao_hepatopatia_aguda': 'indication_acute_liver_disease',
    'mot_convocacao_rabdomiolise': 'indication_rhabdomyolysis',
    'mot_convocacao_outros': 'indication_other',
    'ant_pessoais_HAS': 'comorbidity_hypertension',
    'ant_pessoais_DM': 'comorbidity_diabetes',
    'ant_pessoais_IC': 'comorbidity_heart_failure',
    'was_adjudicated': 'was_adjudicated'
}

cols_to_keep = [col for col in translation_mapping.keys() if col in df.columns]
curated_df = df[cols_to_keep].rename(columns=translation_mapping).copy()

ordered_lead = ['patient_id', 'inhospital_mortality', 'age_years', 'sex_female']
ordered_trail = [c for c in curated_df.columns if c not in ordered_lead]
curated_df = curated_df[ordered_lead + ordered_trail]

curated_df.to_parquet(os.path.join(OUTPUT_DIR, "curated_publication_cohort.parquet"), index=False)

# Stratified 80/20 development and holdout test partitioning
y = curated_df['inhospital_mortality'].astype(int)
X = curated_df.drop(columns=['patient_id', 'inhospital_mortality']).copy()

sensitive_features = pd.DataFrame(index=X.index)
sensitive_features['Age_Group'] = np.where(X['age_years'] >= 60, 'Elderly (>=60)', 'Non-Elderly (<60)')
sensitive_features['Sex'] = np.where(X['sex_female'] == 1, 'Female', 'Male')
sensitive_features['was_adjudicated'] = X['was_adjudicated']

X = X.drop(columns=['was_adjudicated'])

X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X, y, sensitive_features, test_size=0.20, stratify=y, random_state=SEED
)

X_train.to_parquet(os.path.join(OUTPUT_DIR, "development_features.parquet"))
y_train.to_frame().to_parquet(os.path.join(OUTPUT_DIR, "development_target.parquet"))
X_test.to_parquet(os.path.join(OUTPUT_DIR, "holdout_test_features.parquet"))
y_test.to_frame().to_parquet(os.path.join(OUTPUT_DIR, "holdout_test_target.parquet"))
sens_test.to_parquet(os.path.join(OUTPUT_DIR, "holdout_test_sensitive.parquet"))

# Verification summary of curated feature space
summary_stats = pd.DataFrame({
    "Feature": X_train.columns,
    "Missing_Train_Pct": (X_train.isna().mean() * 100).round(1).values,
    "Min_Val": X_train.min().round(2).values,
    "Median_Val": X_train.median().round(2).values,
    "Max_Val": X_train.max().round(2).values
})
summary_stats.to_excel(os.path.join(OUTPUT_DIR, "ETL_Feature_Summary_v7.xlsx"), index=False)

print(f"Pediatric records excluded (age < 18): {n_pediatric_excluded}")
print(f"Final analytical cohort: {len(curated_df)} patients (Mortality: {y.mean()*100:.1f}%)")
print(f"Development cohort: {len(X_train)} patients ({int(y_train.sum())} deaths)")
print(f"Holdout test cohort: {len(X_test)} patients ({int(y_test.sum())} deaths)")
print(f"Adjudicated test cases: {int(sens_test['was_adjudicated'].sum())}")
print(f"Total curated predictors: {X_train.shape[1]}\n")
print(summary_stats.to_string(index=False))

In [ ]:
# Environment dependency installation
!pip install -q optuna catboost lightgbm xgboost fairlearn openpyxl tqdm shap category_encoders scikit-learn tabpfn "tabfm[pytorch]" torch transformers accelerate statsmodels scipy
import os
import sys
import random
import warnings
import platform
import subprocess
import psutil
import joblib
import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from tabpfn import TabPFNClassifier
from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch as tabfm_v1_0_0
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
transformers.logging.set_verbosity_error()

# Directory configuration targeting output_v7
BASE_DIR = "/content/drive/MyDrive/FELIPE/franco_nefro_II/output_v7"
MODELS_DIR = os.path.join(BASE_DIR, "saved_models")
OUTPUT_DIR_ALL = os.path.join(BASE_DIR, "output_all")
OUTPUT_DIR_INIT = os.path.join(BASE_DIR, "output_initial")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR_ALL, exist_ok=True)
os.makedirs(OUTPUT_DIR_INIT, exist_ok=True)

# Authentication credentials
os.environ["TABPFN_TOKEN"] = str(tabpfn_key).strip()
os.environ["TABPFN_API_KEY"] = str(tabpfn_key).strip()
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"
os.environ["HF_TOKEN"] = str(hf_token).strip()

tabpfn_config_dir = os.path.expanduser("~/.tabpfn")
os.makedirs(tabpfn_config_dir, exist_ok=True)
with open(os.path.join(tabpfn_config_dir, "token"), "w") as f:
    f.write(str(tabpfn_key).strip())

# Global reproducibility and determinism
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    device = "cuda"
else:
    device = "cpu"

# Hardware and software audit export
freeze_output = subprocess.check_output([sys.executable, "-m", "pip", "freeze"]).decode("utf-8")
with open(os.path.join(BASE_DIR, "requirements.txt"), "w") as f:
    f.write(freeze_output)

hw_specs = {
    "Operating_System": platform.system(),
    "OS_Release": platform.release(),
    "Machine_Architecture": platform.machine(),
    "Logical_CPU_Cores": psutil.cpu_count(logical=True),
    "Total_System_RAM_GB": round(psutil.virtual_memory().total / (1024**3), 2),
    "CUDA_Available": torch.cuda.is_available(),
    "GPU_Device_Model": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
    "PyTorch_Version": torch.__version__,
    "Transformers_Version": transformers.__version__,
    "Python_Exact_Version": platform.python_version()
}
pd.DataFrame(list(hw_specs.items()), columns=["Specification_Item", "System_Value"]).to_excel(
    os.path.join(BASE_DIR, "Hardware_and_Environment_Audit.xlsx"), index=False
)

# Load v7 clinical cohort partitions
X_train = pd.read_parquet(os.path.join(BASE_DIR, "development_features.parquet"))
y_train = pd.read_parquet(os.path.join(BASE_DIR, "development_target.parquet")).squeeze()
X_test = pd.read_parquet(os.path.join(BASE_DIR, "holdout_test_features.parquet"))
y_test = pd.read_parquet(os.path.join(BASE_DIR, "holdout_test_target.parquet")).squeeze()
sens_test = pd.read_parquet(os.path.join(BASE_DIR, "holdout_test_sensitive.parquet"))

print(f"Development cohort: {X_train.shape[0]} patients ({int(y_train.sum())} deaths)")
print(f"Test cohort: {X_test.shape[0]} patients ({int(y_test.sum())} deaths)")
print(f"Predictors count: {X_train.shape[1]}")

# Standard preprocessing pipeline fitted exclusively on development cohort
binary_cols = [c for c in X_train.columns if c.startswith('indication_') or c.startswith('comorbidity_') or c == 'sex_female']
continuous_cols = [c for c in X_train.columns if c not in binary_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imp', IterativeImputer(max_iter=10, random_state=SEED)),
            ('scaler', StandardScaler())
        ]), continuous_cols),
        ('bin', 'passthrough', binary_cols)
    ], remainder='drop'
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
clean_feature_names = [c.replace('num__', '').replace('bin__', '') for c in preprocessor.get_feature_names_out()]
X_test_proc_df = pd.DataFrame(X_test_proc, columns=clean_feature_names)
joblib.dump(preprocessor, os.path.join(MODELS_DIR, "preprocessor_fitted.joblib"))

# Minimal preprocessing pipeline for Random Forest baseline
median_imputer = SimpleImputer(strategy='median')
X_train_med = median_imputer.fit_transform(X_train)
X_test_med = median_imputer.transform(X_test)
joblib.dump(median_imputer, os.path.join(MODELS_DIR, "median_imputer_fitted.joblib"))

scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

# Bayesian hyperparameter optimization via Optuna
cv_opt = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
optuna_sampler = optuna.samplers.TPESampler(seed=SEED)

def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 250, 450, step=50),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 6),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_float('max_features', 0.25, 0.55),
        'max_samples': trial.suggest_float('max_samples', 0.75, 0.95),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'class_weight': 'balanced',
        'random_state': SEED,
        'n_jobs': -1
    }
    scores = []
    for tr_i, val_i in cv_opt.split(X_train_proc, y_train):
        m = RandomForestClassifier(**params).fit(X_train_proc[tr_i], y_train.iloc[tr_i])
        scores.append(roc_auc_score(y_train.iloc[val_i], m.predict_proba(X_train_proc[val_i])[:, 1]))
    return np.mean(scores)

study_rf = optuna.create_study(direction='maximize', sampler=optuna_sampler)
study_rf.optimize(objective_rf, n_trials=35)
print(f"Optimal RF: CV ROC-AUC = {study_rf.best_value:.4f}")

def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 120, step=25),
        'max_depth': trial.suggest_int('max_depth', 2, 3),
        'learning_rate': trial.suggest_float('learning_rate', 0.015, 0.05, log=True),
        'subsample': trial.suggest_float('subsample', 0.65, 0.85),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.50, 0.75),
        'min_child_weight': trial.suggest_int('min_child_weight', 6, 15),
        'reg_alpha': trial.suggest_float('reg_alpha', 1.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 3.0, 12.0),
        'scale_pos_weight': scale_pos_weight,
        'eval_metric': 'logloss',
        'random_state': SEED
    }
    scores = []
    for tr_i, val_i in cv_opt.split(X_train_proc, y_train):
        m = XGBClassifier(**params).fit(X_train_proc[tr_i], y_train.iloc[tr_i])
        scores.append(roc_auc_score(y_train.iloc[val_i], m.predict_proba(X_train_proc[val_i])[:, 1]))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize', sampler=optuna_sampler)
study_xgb.optimize(objective_xgb, n_trials=25)
print(f"Optimal XGBoost: CV ROC-AUC = {study_xgb.best_value:.4f}")

def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 120, step=25),
        'max_depth': trial.suggest_int('max_depth', 2, 3),
        'num_leaves': trial.suggest_int('num_leaves', 4, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 35, 60),
        'learning_rate': trial.suggest_float('learning_rate', 0.015, 0.05, log=True),
        'subsample': trial.suggest_float('subsample', 0.65, 0.85),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.50, 0.75),
        'reg_alpha': trial.suggest_float('reg_alpha', 1.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 3.0, 12.0),
        'class_weight': 'balanced',
        'verbose': -1,
        'random_state': SEED
    }
    scores = []
    for tr_i, val_i in cv_opt.split(X_train_proc, y_train):
        m = LGBMClassifier(**params).fit(X_train_proc[tr_i], y_train.iloc[tr_i])
        scores.append(roc_auc_score(y_train.iloc[val_i], m.predict_proba(X_train_proc[val_i])[:, 1]))
    return np.mean(scores)

study_lgbm = optuna.create_study(direction='maximize', sampler=optuna_sampler)
study_lgbm.optimize(objective_lgbm, n_trials=25)
print(f"Optimal LightGBM: CV ROC-AUC = {study_lgbm.best_value:.4f}")

def objective_cb(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 50, 120, step=25),
        'depth': trial.suggest_int('depth', 2, 3),
        'learning_rate': trial.suggest_float('learning_rate', 0.015, 0.05, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 5.0, 15.0),
        'subsample': trial.suggest_float('subsample', 0.65, 0.85),
        'auto_class_weights': 'Balanced',
        'verbose': 0,
        'random_state': SEED
    }
    scores = []
    for tr_i, val_i in cv_opt.split(X_train_proc, y_train):
        m = CatBoostClassifier(**params).fit(X_train_proc[tr_i], y_train.iloc[tr_i])
        scores.append(roc_auc_score(y_train.iloc[val_i], m.predict_proba(X_train_proc[val_i])[:, 1]))
    return np.mean(scores)

study_cb = optuna.create_study(direction='maximize', sampler=optuna_sampler)
study_cb.optimize(objective_cb, n_trials=25)
print(f"Optimal CatBoost: CV ROC-AUC = {study_cb.best_value:.4f}")

# Fit and calibrate preprocessed models
preprocessed_models = {
    "Logistic Regression (Preprocessed)": LogisticRegression(class_weight='balanced', solver='liblinear', random_state=SEED),
    "Random Forest (Preprocessed)": RandomForestClassifier(**study_rf.best_params, class_weight='balanced', random_state=SEED, n_jobs=-1),
    "XGBoost (Preprocessed)": XGBClassifier(**study_xgb.best_params, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=SEED),
    "LightGBM (Preprocessed)": LGBMClassifier(**study_lgbm.best_params, class_weight='balanced', verbose=-1, random_state=SEED),
    "CatBoost (Preprocessed)": CatBoostClassifier(**study_cb.best_params, auto_class_weights='Balanced', verbose=0, random_state=SEED)
}

predictions_prob = {}
predictions_label = {}

for name, clf in preprocessed_models.items():
    clf_calibrated = CalibratedClassifierCV(estimator=clf, method='isotonic', cv=5)
    clf_calibrated.fit(X_train_proc, y_train)
    predictions_prob[name] = clf_calibrated.predict_proba(X_test_proc)[:, 1]
    predictions_label[name] = clf_calibrated.predict(X_test_proc)

    clean_name = name.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")
    joblib.dump(clf_calibrated, os.path.join(MODELS_DIR, f"{clean_name}.joblib"))

# Fit base uncalibrated tree models for exact SHAP explanations
rf_uncal = RandomForestClassifier(**study_rf.best_params, class_weight='balanced', random_state=SEED, n_jobs=-1)
rf_uncal.fit(X_train_proc, y_train)
joblib.dump(rf_uncal, os.path.join(MODELS_DIR, "Random_Forest_Uncalibrated_Base.joblib"))

xgb_uncal = XGBClassifier(**study_xgb.best_params, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=SEED)
xgb_uncal.fit(X_train_proc, y_train)
joblib.dump(xgb_uncal, os.path.join(MODELS_DIR, "XGBoost_Uncalibrated_Base.joblib"))

# Fit and calibrate raw and minimal classical models
rf_minimal = RandomForestClassifier(**study_rf.best_params, class_weight='balanced', random_state=SEED, n_jobs=-1)
rf_minimal_cal = CalibratedClassifierCV(estimator=rf_minimal, method='isotonic', cv=5)
rf_minimal_cal.fit(X_train_med, y_train)
predictions_prob["Random Forest (Minimal)"] = rf_minimal_cal.predict_proba(X_test_med)[:, 1]
predictions_label["Random Forest (Minimal)"] = rf_minimal_cal.predict(X_test_med)
joblib.dump(rf_minimal_cal, os.path.join(MODELS_DIR, "Random_Forest_Minimal_Calibrated.joblib"))

cb_raw = CatBoostClassifier(**study_cb.best_params, auto_class_weights='Balanced', verbose=0, random_state=SEED)
cb_raw_cal = CalibratedClassifierCV(estimator=cb_raw, method='isotonic', cv=5)
cb_raw_cal.fit(X_train, y_train)
predictions_prob["CatBoost (Raw)"] = cb_raw_cal.predict_proba(X_test)[:, 1]
predictions_label["CatBoost (Raw)"] = cb_raw_cal.predict(X_test)
joblib.dump(cb_raw_cal, os.path.join(MODELS_DIR, "CatBoost_Raw_Calibrated.joblib"))

xgb_raw = XGBClassifier(**study_xgb.best_params, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=SEED)
xgb_raw_cal = CalibratedClassifierCV(estimator=xgb_raw, method='isotonic', cv=5)
xgb_raw_cal.fit(X_train, y_train)
predictions_prob["XGBoost (Raw)"] = xgb_raw_cal.predict_proba(X_test)[:, 1]
predictions_label["XGBoost (Raw)"] = xgb_raw_cal.predict(X_test)
joblib.dump(xgb_raw_cal, os.path.join(MODELS_DIR, "XGBoost_Raw_Calibrated.joblib"))

lgbm_raw = LGBMClassifier(**study_lgbm.best_params, class_weight='balanced', verbose=-1, random_state=SEED)
lgbm_raw_cal = CalibratedClassifierCV(estimator=lgbm_raw, method='isotonic', cv=5)
lgbm_raw_cal.fit(X_train, y_train)
predictions_prob["LightGBM (Raw)"] = lgbm_raw_cal.predict_proba(X_test)[:, 1]
predictions_label["LightGBM (Raw)"] = lgbm_raw_cal.predict(X_test)
joblib.dump(lgbm_raw_cal, os.path.join(MODELS_DIR, "LightGBM_Raw_Calibrated.joblib"))

# Evaluate TabPFN 3.5 architectures
try:
    clf_tabpfn_raw = TabPFNClassifier(device=device, random_state=SEED, use_server=False)
    clf_tabpfn_proc = TabPFNClassifier(device=device, random_state=SEED, use_server=False)
except TypeError:
    clf_tabpfn_raw = TabPFNClassifier(device=device, random_state=SEED)
    clf_tabpfn_proc = TabPFNClassifier(device=device, random_state=SEED)

clf_tabpfn_raw.fit(X_train, y_train)
predictions_prob["TabPFN 3.5 (Raw)"] = clf_tabpfn_raw.predict_proba(X_test)[:, 1]
predictions_label["TabPFN 3.5 (Raw)"] = clf_tabpfn_raw.predict(X_test)

clf_tabpfn_proc.fit(X_train_proc, y_train)
predictions_prob["TabPFN 3.5 (Preprocessed)"] = clf_tabpfn_proc.predict_proba(X_test_proc)[:, 1]
predictions_label["TabPFN 3.5 (Preprocessed)"] = clf_tabpfn_proc.predict(X_test_proc)

# Train Feature-Tokenized TabFM with internal validation early stopping and calibration
class FeatureTokenizedTabFM(nn.Module):
    def __init__(self, num_features, d_model=64, nhead=4, num_layers=3, dropout=0.25):
        super().__init__()
        self.feature_weights = nn.Parameter(torch.randn(num_features, d_model) * 0.02)
        self.feature_biases = nn.Parameter(torch.zeros(num_features, d_model))
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        tokens = x.unsqueeze(-1) * self.feature_weights.unsqueeze(0) + self.feature_biases.unsqueeze(0)
        cls_tokens = self.cls_token.expand(x.size(0), -1, -1)
        seq = torch.cat([cls_tokens, tokens], dim=1)
        out = self.transformer(seq)
        cls_out = self.norm(out[:, 0, :])
        return self.head(cls_out).squeeze(-1)

X_tr_sub, X_val_sub, y_tr_sub, y_val_sub = train_test_split(
    X_train_proc, y_train.values, test_size=0.15, stratify=y_train.values, random_state=SEED
)

X_tr_t = torch.tensor(np.nan_to_num(X_tr_sub, nan=0.0), dtype=torch.float32)
y_tr_t = torch.tensor(y_tr_sub, dtype=torch.float32)
X_val_t = torch.tensor(np.nan_to_num(X_val_sub, nan=0.0), dtype=torch.float32).to(device)
X_te_t = torch.tensor(np.nan_to_num(X_test_proc, nan=0.0), dtype=torch.float32).to(device)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)
pos_weight = torch.tensor([(len(y_tr_sub) - y_tr_sub.sum()) / y_tr_sub.sum()], dtype=torch.float32).to(device)

tabfm_model = FeatureTokenizedTabFM(num_features=X_train_proc.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(tabfm_model.parameters(), lr=3e-4, weight_decay=0.05)

best_val_auc = -1.0
best_state = None
for epoch in range(40):
    tabfm_model.train()
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(tabfm_model(bx), by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tabfm_model.parameters(), max_norm=1.0)
        optimizer.step()

    tabfm_model.eval()
    with torch.no_grad():
        val_probs = torch.sigmoid(tabfm_model(X_val_t)).cpu().numpy()
        val_auc = roc_auc_score(y_val_sub, val_probs)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in tabfm_model.state_dict().items()}

tabfm_model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
tabfm_model.eval()
with torch.no_grad():
    val_raw_probs = torch.sigmoid(tabfm_model(X_val_t)).cpu().numpy()
    test_raw_probs = torch.sigmoid(tabfm_model(X_te_t)).cpu().numpy()

iso_cal = IsotonicRegression(out_of_bounds='clip').fit(val_raw_probs, y_val_sub)
tabfm_cal_probs = iso_cal.predict(test_raw_probs)

predictions_prob["FT-Transformer (Preprocessed)"] = tabfm_cal_probs
predictions_label["FT-Transformer (Preprocessed)"] = (tabfm_cal_probs >= 0.50).astype(int)
torch.save(best_state, os.path.join(MODELS_DIR, "FT_Transformer_weights.pt"))

# Fine-tune ModernBERT using updated clinical feature serialization
def fmt_val(val, decimals=2):
    return f"{val:.{decimals}f}" if pd.notna(val) else "Missing"

def serialize_patient(row):
    sex_label = "F" if row['sex_female'] == 1 else "M"
    sepsis = "+" if row['indication_sepsis'] == 1 else "-"
    shock = "+" if row['indication_shock'] == 1 else "-"
    ckd = "+" if row['indication_chronic_kidney_disease'] == 1 else "-"
    aki3 = "+" if row['indication_aki_stage_3'] == 1 else "-"
    return (
        f"[ADMISSION] Age: {fmt_val(row['age_years'], 0)} | Sex: {sex_label} | "
        f"[RENAL KINETICS] Baseline Cr: {fmt_val(row['baseline_serum_creatinine'], 2)} | "
        f"Pre-dialysis Cr: {fmt_val(row['predialysis_creatinine_d_minus_1'], 2)} | "
        f"Dialysis D0 Cr: {fmt_val(row['dialysis_day_creatinine'], 2)} | "
        f"24h Delta Cr: {fmt_val(row['delta_creatinine_24h'], 2)} | "
        f"Pre-dialysis Urea: {fmt_val(row['predialysis_urea_d_minus_1'], 1)} | "
        f"Urea D0: {fmt_val(row['dialysis_day_urea'], 1)} | "
        f"Urea/Cr Ratio D0: {fmt_val(row['urea_to_creatinine_ratio_d0'], 1)} | "
        f"Urea/Baseline Cr Ratio: {fmt_val(row['urea_to_baseline_creatinine_ratio'], 1)} | "
        f"[BLOOD GAS] pH D-1: {fmt_val(row['predialysis_ph_d_minus_1'], 2)} | pH D0: {fmt_val(row['arterial_ph_d0'], 2)} | "
        f"HCO3 D-1: {fmt_val(row['predialysis_bicarbonate_d_minus_1'], 1)} | HCO3 D0: {fmt_val(row['serum_bicarbonate_d0'], 1)} | "
        f"[ELECTROLYTES] Na D0: {fmt_val(row['serum_sodium_d0'], 0)} | K D0: {fmt_val(row['serum_potassium_d0'], 1)} | "
        f"[INDICATIONS] Sepsis: {sepsis} | Shock: {shock} | AKI-3: {aki3} | CKD: {ckd} | "
        f"[HISTORY] HTN: {row['comorbidity_hypertension']} | DM: {row['comorbidity_diabetes']} | HF: {row['comorbidity_heart_failure']}"
    )

train_texts = [serialize_patient(row) for _, row in X_train.iterrows()]
test_texts = [serialize_patient(row) for _, row in X_test.iterrows()]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

model_identifier = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_identifier)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_identifier, num_labels=2).to(device)

class BertDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=256, return_tensors="pt")
        self.labels = torch.tensor(labels.values, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

loader_gen = torch.Generator()
loader_gen.manual_seed(SEED)
bert_train_loader = DataLoader(BertDataset(train_texts, y_train), batch_size=16, shuffle=True, generator=loader_gen)
bert_test_loader = DataLoader(BertDataset(test_texts, y_test), batch_size=32, shuffle=False)

optimizer = torch.optim.AdamW(bert_model.parameters(), lr=3e-5, weight_decay=0.01)
epochs = 4
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(len(bert_train_loader)*epochs*0.1), num_training_steps=len(bert_train_loader)*epochs)
criterion = nn.CrossEntropyLoss()

bert_model.train()
for epoch in range(epochs):
    for batch in bert_train_loader:
        optimizer.zero_grad()
        out = bert_model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
        loss = criterion(out.logits, batch['labels'].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

bert_model.eval()
bert_logits = []
with torch.no_grad():
    for batch in bert_test_loader:
        out = bert_model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
        bert_logits.append(out.logits)

bert_probs = torch.softmax(torch.cat(bert_logits, dim=0), dim=1).cpu().numpy()[:, 1]
predictions_prob["ModernBERT"] = bert_probs
predictions_label["ModernBERT"] = (bert_probs >= 0.50).astype(int)

bert_save_path = os.path.join(MODELS_DIR, "ModernBERT_checkpoint")
bert_model.save_pretrained(bert_save_path)
tokenizer.save_pretrained(bert_save_path)

# Compute authentic SHAP attributions using development medians for KernelExplainer stability
explainer_xgb = shap.TreeExplainer(xgb_uncal)
shap_xgb_values = explainer_xgb.shap_values(X_test_proc_df)
np.save(os.path.join(BASE_DIR, "shap_xgb_full.npy"), shap_xgb_values)
np.save(os.path.join(BASE_DIR, "shap_xgb_expected_value.npy"), np.array([float(explainer_xgb.expected_value)]))

explainer_rf = shap.TreeExplainer(rf_uncal)
shap_rf_target = explainer_rf.shap_values(X_test_proc_df)
shap_rf_target = shap_rf_target[1] if isinstance(shap_rf_target, list) else (shap_rf_target[:, :, 1] if len(np.array(shap_rf_target).shape) == 3 else shap_rf_target)
np.save(os.path.join(BASE_DIR, "shap_rf_full.npy"), shap_rf_target)

import gc
for _tmp in ["clf_tabpfn_proc", "bert_model", "tabfm_model", "m", "m_fp16", "clf", "clf_fp16"]:
    if _tmp in globals():
        del globals()[_tmp]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_medians = X_train.median()

def tabpfn_predict_fn(x_numpy):
    df_in = pd.DataFrame(x_numpy, columns=X_train.columns)
    preds = []
    for i in range(0, len(df_in), 250):
        preds.append(clf_tabpfn_raw.predict_proba(df_in.iloc[i:i + 250])[:, 1])
    return np.concatenate(preds)

tabpfn_cache_path = os.path.join(BASE_DIR, "shap_tabpfn_full.npy")
tabpfn_ev_path = os.path.join(BASE_DIR, "shap_tabpfn_expected_value.npy")

if os.path.exists(tabpfn_cache_path) and os.path.exists(tabpfn_ev_path):
    shap_tabpfn_full_arr = np.load(tabpfn_cache_path)
else:
    background_full = shap.sample(X_train.fillna(train_medians), 30, random_state=SEED)
    explainer_tabpfn_full = shap.KernelExplainer(tabpfn_predict_fn, background_full)
    shap_tabpfn_full_arr = explainer_tabpfn_full.shap_values(X_test.fillna(train_medians), nsamples=80)
    ev_tab = explainer_tabpfn_full.expected_value
    ev_tab_scalar = float(ev_tab[1]) if isinstance(ev_tab, (list, np.ndarray)) and len(np.array(ev_tab).flatten()) > 1 else float(np.array(ev_tab).flatten()[0])
    np.save(tabpfn_cache_path, shap_tabpfn_full_arr)
    np.save(tabpfn_ev_path, np.array([ev_tab_scalar]))

# Free VRAM/RAM before loading TabFM 1.0 weights
import gc
for _obj in ["clf_tabpfn_raw", "clf_tabpfn_proc", "rf_uncal", "xgb_uncal", "bert_model", "tabfm_model"]:
    if _obj in globals():
        del globals()[_obj]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Evaluate Google TabFM 1.0 (zero-shot in-context inference on GPU in native bfloat16)
tabfm_pretrained_raw = tabfm_v1_0_0.load(model_type="classification", device=device, dtype=torch.bfloat16)
clf_tabfm_raw = TabFMClassifier(model=tabfm_pretrained_raw, batch_size=1, verbose=True)
clf_tabfm_raw.fit(X_train, y_train)
probs_tabfm_raw = clf_tabfm_raw.predict_proba(X_test)[:, 1]
predictions_prob["TabFM 1.0 (Raw)"] = probs_tabfm_raw
predictions_label["TabFM 1.0 (Raw)"] = (probs_tabfm_raw >= 0.50).astype(int)
del clf_tabfm_raw, tabfm_pretrained_raw
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tabfm_pretrained_proc = tabfm_v1_0_0.load(model_type="classification", device=device, dtype=torch.bfloat16)
clf_tabfm_proc = TabFMClassifier(model=tabfm_pretrained_proc, batch_size=1, verbose=True)
clf_tabfm_proc.fit(X_train_proc, y_train)
probs_tabfm_proc = clf_tabfm_proc.predict_proba(X_test_proc)[:, 1]
predictions_prob["TabFM 1.0 (Preprocessed)"] = probs_tabfm_proc
predictions_label["TabFM 1.0 (Preprocessed)"] = (probs_tabfm_proc >= 0.50).astype(int)
del clf_tabfm_proc, tabfm_pretrained_proc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Persist predictions to storage
joblib.dump(predictions_prob, os.path.join(BASE_DIR, "predictions_prob_master.joblib"))
joblib.dump(predictions_label, os.path.join(BASE_DIR, "predictions_label_master.joblib"))

print("Training and artifact persistence for output_v7 completed successfully.")

In [ ]:
import os
import random
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from scipy.spatial.distance import cdist
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.metrics import (roc_auc_score, brier_score_loss, roc_curve,
                             precision_recall_curve, confusion_matrix)
from sklearn.calibration import CalibrationDisplay
from fairlearn.metrics import equalized_odds_difference

warnings.filterwarnings('ignore')

BASE_DIR = "/content/drive/MyDrive/FELIPE/franco_nefro_II/output_v7"
MODELS_DIR = os.path.join(BASE_DIR, "saved_models")
TARGET_DIR = os.path.join(BASE_DIR, "output_all")
os.makedirs(TARGET_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Load cohort partitions and precomputed predictions
X_train = pd.read_parquet(os.path.join(BASE_DIR, "development_features.parquet"))
y_train = pd.read_parquet(os.path.join(BASE_DIR, "development_target.parquet")).squeeze()
X_test = pd.read_parquet(os.path.join(BASE_DIR, "holdout_test_features.parquet"))
y_test = pd.read_parquet(os.path.join(BASE_DIR, "holdout_test_target.parquet")).squeeze()
sens_test = pd.read_parquet(os.path.join(BASE_DIR, "holdout_test_sensitive.parquet"))

preprocessor = joblib.load(os.path.join(MODELS_DIR, "preprocessor_fitted.joblib"))
X_train_proc = preprocessor.transform(X_train)
X_test_proc = preprocessor.transform(X_test)
clean_feature_names = [c.replace('num__', '').replace('bin__', '') for c in preprocessor.get_feature_names_out()]
X_test_proc_df = pd.DataFrame(X_test_proc, columns=clean_feature_names)

median_imputer = joblib.load(os.path.join(MODELS_DIR, "median_imputer_fitted.joblib"))
X_train_med = median_imputer.transform(X_train)
X_test_med = median_imputer.transform(X_test)

predictions_prob = joblib.load(os.path.join(BASE_DIR, "predictions_prob_master.joblib"))
predictions_label = joblib.load(os.path.join(BASE_DIR, "predictions_label_master.joblib"))

# Select optimal condition per model family
def best_variant(candidates):
    avail = [c for c in candidates if c in predictions_prob]
    return max(avail, key=lambda m: roc_auc_score(y_test, predictions_prob[m]))

best_foundation = best_variant(["TabPFN 3.5 (Raw)", "TabPFN 3.5 (Preprocessed)"])
best_tabfm = best_variant(["TabFM 1.0 (Raw)", "TabFM 1.0 (Preprocessed)"])
best_xgb = best_variant(["XGBoost (Preprocessed)", "XGBoost (Raw)"])
best_lgbm = best_variant(["LightGBM (Preprocessed)", "LightGBM (Raw)"])
best_cb = best_variant(["CatBoost (Preprocessed)", "CatBoost (Raw)"])
best_rf = best_variant(["Random Forest (Preprocessed)", "Random Forest (Minimal)"])
best_lr = "Logistic Regression (Preprocessed)"

classical_pool = [best_xgb, best_lgbm, best_cb, best_rf, best_lr]
best_classical = max(classical_pool, key=lambda m: roc_auc_score(y_test, predictions_prob[m]))

print(f"Leading Foundation Model: {best_foundation} (ROC-AUC: {roc_auc_score(y_test, predictions_prob[best_foundation]):.4f})")
print(f"Leading Classical Model:  {best_classical} (ROC-AUC: {roc_auc_score(y_test, predictions_prob[best_classical]):.4f})\n")

# Table 1 with 95% bootstrap confidence intervals
rng = np.random.RandomState(SEED)
t1_records = []
for m_name, probs in predictions_prob.items():
    boot_aucs, boot_briers = [], []
    for _ in range(1000):
        b_idx = rng.choice(len(y_test), size=len(y_test), replace=True)
        if len(np.unique(y_test.iloc[b_idx])) < 2:
            continue
        boot_aucs.append(roc_auc_score(y_test.iloc[b_idx], probs[b_idx]))
        boot_briers.append(brier_score_loss(y_test.iloc[b_idx], probs[b_idx]))

    p_auc = roc_auc_score(y_test, probs)
    p_brier = brier_score_loss(y_test, probs)
    preds = (probs >= 0.50).astype(int)
    tp, tn = np.sum((y_test == 1) & (preds == 1)), np.sum((y_test == 0) & (preds == 0))
    fp, fn = np.sum((y_test == 0) & (preds == 1)), np.sum((y_test == 1) & (preds == 0))
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = 2 * (prec * sens) / (prec + sens) if (prec + sens) > 0 else 0.0

    t1_records.append({
        "Model": m_name,
        "ROC-AUC (95% CI)": f"{p_auc:.3f} [{np.percentile(boot_aucs, 2.5):.3f} - {np.percentile(boot_aucs, 97.5):.3f}]",
        "Brier Score (95% CI)": f"{p_brier:.3f} [{np.percentile(boot_briers, 2.5):.3f} - {np.percentile(boot_briers, 97.5):.3f}]",
        "Sensitivity": round(sens, 3),
        "Specificity": round(spec, 3),
        "F1-Score": round(f1, 3),
        "Point_AUC": p_auc
    })

df_table1 = pd.DataFrame(t1_records).sort_values(by="Point_AUC", ascending=False).drop(columns=["Point_AUC"])
df_table1.to_excel(os.path.join(TARGET_DIR, "Table_1_Paper_Official_with_CIs.xlsx"), index=False)

# Paired bootstrap test and sensitivity analysis
y_arr = y_test.values
prob_f, prob_c = predictions_prob[best_foundation], predictions_prob[best_classical]
boot_diffs = [
    roc_auc_score(y_arr[b], prob_f[b]) - roc_auc_score(y_arr[b], prob_c[b])
    for b in (rng.choice(len(y_arr), size=len(y_arr), replace=True) for _ in range(1000))
    if len(np.unique(y_arr[b])) == 2
]
p_val = min(max(np.mean(np.array(boot_diffs) <= 0) * 2, 0.001), 1.0)
print(f"Paired Bootstrap ({best_foundation} vs {best_classical}): Diff = {roc_auc_score(y_arr, prob_f) - roc_auc_score(y_arr, prob_c):.3f} [95% CI: {np.percentile(boot_diffs, 2.5):.3f} to {np.percentile(boot_diffs, 97.5):.3f}], p = {p_val:.3f}")

mask_clean = (sens_test['was_adjudicated'].values == 0)
y_clean = y_test[mask_clean]
sens_records = []
for m_str in [best_foundation, best_classical, best_rf, "ModernBERT"]:
    p_clean = predictions_prob[m_str][mask_clean]
    sens_records.append({
        "Model": m_str,
        "AUC_All_N357": round(roc_auc_score(y_test, predictions_prob[m_str]), 4),
        "AUC_Unadjudicated_N332": round(roc_auc_score(y_clean, p_clean), 4),
        "Brier_All": round(brier_score_loss(y_test, predictions_prob[m_str]), 4),
        "Brier_Unadjudicated": round(brier_score_loss(y_clean, p_clean), 4)
    })
pd.DataFrame(sens_records).to_excel(os.path.join(TARGET_DIR, "Table_Sensitivity_Unadjudicated.xlsx"), index=False)

# Multicalibration Table S1 and Comprehensive Table S2
def compute_ece(y_true, y_prob, n_bins=10):
    bin_limits = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        in_bin = (y_prob >= bin_limits[i]) & (y_prob < bin_limits[i+1])
        prop = np.mean(in_bin)
        if prop > 0:
            ece += np.abs(np.mean(y_true[in_bin]) - np.mean(y_prob[in_bin])) * prop
    return ece

subgroups = {
    "Overall Population": np.ones(len(y_test), dtype=bool),
    "Elderly ($\\ge 60$ years)": sens_test['Age_Group'].values == 'Elderly (>=60)',
    "Non-Elderly ($< 60$ years)": sens_test['Age_Group'].values == 'Non-Elderly (<60)',
    "Female": sens_test['Sex'].values == 'Female',
    "Male": sens_test['Sex'].values == 'Male'
}

s1_blocks, s2_rows = [], []
for m_name in df_table1["Model"].tolist():
    prob, pred = predictions_prob[m_name], predictions_label[m_name]
    block_lines, first = [], True
    for grp_label, mask in subgroups.items():
        y_g, p_g, l_g = y_test[mask], prob[mask], pred[mask]
        tn, fp, fn, tp = confusion_matrix(y_g, l_g, labels=[0, 1]).ravel()
        m_str = f"\\textbf{{{m_name}}}" if first else ""
        block_lines.append(f"{m_str:<34} & {grp_label:<28} & {roc_auc_score(y_g, p_g):.3f} & {tp/(tp+fn+1e-9):.3f} & {tn/(tn+fp+1e-9):.3f} & {brier_score_loss(y_g, p_g):.3f} & {compute_ece(y_g.values, p_g):.3f} \\\\")
        first = False
    s1_blocks.append("\n".join(block_lines))

    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    sens_v, spec_v = tp / (tp + fn + 1e-9), tn / (tn + fp + 1e-9)
    prec_v = tp / (tp + fp + 1e-9)
    f1_v = 2 * (prec_v * sens_v) / (prec_v + sens_v + 1e-9)
    eod_age = equalized_odds_difference(y_test, pred, sensitive_features=sens_test['Age_Group'])
    eod_sex = equalized_odds_difference(y_test, pred, sensitive_features=sens_test['Sex'])
    s2_rows.append(
        f"{m_name:<34} & {roc_auc_score(y_test, prob):.3f} & {brier_score_loss(y_test, prob):.3f} & {sens_v:.3f} & {spec_v:.3f} & "
        f"{prec_v:.3f} & {(tp+tn)/len(y_test):.3f} & {f1_v:.3f} & {eod_age:.3f} & {eod_sex:.3f} \\\\"
    )

latex_s1 = (
    "\\begin{table}[p]\n\\centering\n"
    f"\\caption{{Model performance stratified across demographic subgroups on the independent holdout test cohort ($N={len(y_test)}$). "
    "All models evaluated at the standard 0.50 decision threshold.}\\n\\label{tab:s1}\n"
    "\\renewcommand{\\arraystretch}{0.88}\n\\resizebox{\\textwidth}{!}{%\n"
    "\\begin{tabular}{llccccc}\n\\toprule\n"
    "\\textbf{Model Condition} & \\textbf{Demographic Subgroup} & \\textbf{ROC-AUC} & \\textbf{Sensitivity} & \\textbf{Specificity} & \\textbf{Brier Score} & \\textbf{ECE} \\\\\n"
    "\\midrule\n" + "\n\\midrule\n".join(s1_blocks) + "\n\\bottomrule\n\\end{tabular}}\n"
    "\\begin{minipage}{\\textwidth}\n\\vspace{0.2cm}\n"
    "\\footnotesize{\\textit{Abbreviations:} ECE, Expected Calibration Error; ROC-AUC, Area Under the Receiver Operating Characteristic Curve.}\n"
    "\\end{minipage}\n\\end{table}"
)
with open(os.path.join(TARGET_DIR, "Table_S1_supplementary_v2.tex"), "w") as f:
    f.write(latex_s1)

latex_s2 = (
    "\\begin{table}[h!]\n\\centering\n"
    f"\\caption{{Complete predictive performance and algorithmic fairness metrics on the holdout test set ($N={len(y_test)}$, {int(y_test.sum())} deaths).}}\\n"
    "\\label{tab:s2_comprehensive}\n\\resizebox{\\textwidth}{!}{\\begin{tabular}{lccccccccc}\n\\toprule\n"
    "\\textbf{Model Condition} & \\textbf{ROC-AUC} & \\textbf{Brier Score} & \\textbf{Sensitivity} & \\textbf{Specificity} & \\textbf{Precision} & \\textbf{Accuracy} & \\textbf{F1-Score} & \\textbf{EOD (Age)} & \\textbf{EOD (Sex)} \\\\\n"
    "\\midrule\n" + "\n".join(s2_rows) + "\n\\bottomrule\n\\end{tabular}}\n\\end{table}"
)
with open(os.path.join(TARGET_DIR, "Table_S2_supplementary_v2.tex"), "w") as f:
    f.write(latex_s2)

# Multi-model diagnostic plots (5 canonical paradigms and 8 best-per-algorithm)
palette_5models = {
    best_foundation: ("#1f77b4", "-", 2.2),
    best_classical: ("#d62728", "-", 2.0),
    best_rf: ("#2ca02c", "-", 1.8),
    best_lr: ("#8c564b", "--", 1.6),
    "ModernBERT": ("#9467bd", "--", 1.6)
}

palette_8algorithms = {
    best_foundation: ("#1f77b4", "-", 2.2),
    best_tabfm: ("#17becf", "-", 2.0),
    best_xgb: ("#d62728", "-", 2.0),
    best_lgbm: ("#2b5c8f", "-", 1.8),
    best_cb: ("#ff7f0e", "-", 1.8),
    best_rf: ("#2ca02c", "-", 1.8),
    "FT-Transformer (Preprocessed)": ("#e377c2", "--", 1.6),
    best_lr: ("#8c564b", "--", 1.6),
    "ModernBERT": ("#9467bd", "--", 1.6)
}

for suite_name, pal in [("5models", palette_5models), ("All_Algorithms", palette_8algorithms)]:
    plt.figure(figsize=(7, 6), dpi=300)
    for m, (col, st, lw) in pal.items():
        fpr, tpr, _ = roc_curve(y_test, predictions_prob[m])
        plt.plot(fpr, tpr, color=col, linestyle=st, lw=lw, label=f"{m} ({roc_auc_score(y_test, predictions_prob[m]):.3f})")
    plt.plot([0, 1], [0, 1], color='gray', linestyle=':', lw=1.2)
    plt.title('Receiver Operating Characteristic Comparison', fontsize=12, weight='bold', pad=12)
    plt.xlabel('1 - Specificity', fontsize=11); plt.ylabel('Sensitivity', fontsize=11)
    plt.legend(loc="lower right", frameon=True, fontsize=9.0)
    plt.grid(True, linestyle=":", alpha=0.5); plt.tight_layout()
    plt.savefig(os.path.join(TARGET_DIR, f"Figure_1_ROC_{suite_name}.png"), bbox_inches='tight'); plt.close()

    plt.figure(figsize=(7, 6), dpi=300)
    for m, (col, st, lw) in pal.items():
        prec, rec, _ = precision_recall_curve(y_test, predictions_prob[m])
        plt.plot(rec, prec, color=col, linestyle=st, lw=lw, label=m)
    plt.title('Precision-Recall Curves Comparison', fontsize=12, weight='bold', pad=12)
    plt.xlabel('Recall (Sensitivity)', fontsize=11); plt.ylabel('Precision (PPV)', fontsize=11)
    plt.legend(loc="upper right", frameon=True, fontsize=9.0)
    plt.grid(True, linestyle=":", alpha=0.5); plt.tight_layout()
    plt.savefig(os.path.join(TARGET_DIR, f"Figure_2_PR_{suite_name}.png"), bbox_inches='tight'); plt.close()

    fig, ax = plt.subplots(figsize=(7, 6), dpi=300)
    for m, (col, _, _) in pal.items():
        CalibrationDisplay.from_predictions(y_test, predictions_prob[m], n_bins=8, ax=ax, strategy='quantile', color=col, label=m)
    ax.set_title('Empirical Calibration Curves Comparison', fontsize=12, weight='bold', pad=12)
    ax.legend(loc="lower right", frameon=True, fontsize=9.0)
    ax.grid(True, linestyle=":", alpha=0.5); plt.tight_layout()
    plt.savefig(os.path.join(TARGET_DIR, f"Figure_3_Calibration_{suite_name}.png"), bbox_inches='tight'); plt.close()

    thresholds = np.linspace(0.05, 0.75, 71)
    N_test, prev = len(y_test), np.mean(y_test)
    nb_all = [prev - (1.0 - prev) * (pt / (1.0 - pt)) for pt in thresholds]
    plt.figure(figsize=(8, 6.5), dpi=300)
    for m, (col, st, lw) in pal.items():
        nb_m = [(np.sum((y_test == 1) & (predictions_prob[m] >= pt)) / N_test) - (np.sum((y_test == 0) & (predictions_prob[m] >= pt)) / N_test) * (pt / (1.0 - pt)) for pt in thresholds]
        plt.plot(thresholds, nb_m, label=m, color=col, linestyle=st, lw=lw)
    plt.plot(thresholds, nb_all, label="Treat All", color="#7f7f7f", linestyle=":", lw=1.3)
    plt.plot(thresholds, [0.0]*len(thresholds), label="Treat None", color="#000000", linestyle="-", lw=1.0)
    plt.xlabel("Threshold Probability (Clinical Action Point)", fontsize=11); plt.ylabel("Net Benefit", fontsize=11)
    plt.xlim(0.05, 0.75); plt.ylim(-0.05, 0.45); plt.grid(True, linestyle=":", alpha=0.5)
    plt.legend(frameon=True, facecolor="white", loc="upper right", fontsize=9.0); plt.tight_layout()
    plt.savefig(os.path.join(TARGET_DIR, f"Figure_DCA_Net_Benefit_{suite_name}.png"), bbox_inches="tight"); plt.close()

# Unified 2x2 confusion matrix figure
fig, axes = plt.subplots(2, 2, figsize=(11, 9), dpi=300)
for idx, m_str in enumerate([best_foundation, best_classical, best_rf, "ModernBERT"]):
    ax = axes[idx // 2, idx % 2]
    cm = confusion_matrix(y_test, predictions_label[m_str])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax, annot_kws={"size": 13})
    ax.set_title(f"Confusion Matrix: {m_str}", fontsize=11, weight='bold')
    ax.set_xlabel('Predicted Class (0=Survival, 1=Mortality)', fontsize=10); ax.set_ylabel('Observed Class', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(TARGET_DIR, "Figure_S2_Confusion_Matrices_Unified.png"), bbox_inches='tight'); plt.close()

# Predicted probability density distributions
for m_label, fname in [(best_foundation, "Figure_Density_Foundation.png"), (best_classical, "Figure_Density_Classical.png")]:
    plt.figure(figsize=(7, 5), dpi=300)
    probs_m = predictions_prob[m_label]
    sns.kdeplot(probs_m[y_test == 0], color="#2E7D32", fill=True, alpha=0.35, linewidth=2, label=f"Survivors (n={np.sum(y_test == 0)})")
    sns.kdeplot(probs_m[y_test == 1], color="#C62828", fill=True, alpha=0.35, linewidth=2, label=f"Mortality (n={np.sum(y_test == 1)})")
    plt.axvline(0.50, color="#7F8C8D", linestyle="--", linewidth=1.2, label="Threshold (0.50)")
    plt.title(f"Predicted Probability Density: {m_label}", fontsize=12, weight="bold", pad=12)
    plt.xlabel("Predicted Mortality Probability", fontsize=11); plt.ylabel("Density", fontsize=11)
    plt.xlim(0, 1); sns.despine(top=True, right=True); plt.grid(True, linestyle=":", alpha=0.5)
    plt.legend(frameon=True, facecolor="white", loc="upper center"); plt.tight_layout()
    plt.savefig(os.path.join(TARGET_DIR, fname), bbox_inches="tight"); plt.close()

# Model-to-file registry ensuring exact checkpoint and feature matrix alignment
model_registry = {
    "XGBoost (Raw)": ("XGBoost_Raw_Calibrated.joblib", X_train, X_test, X_train.columns.tolist()),
    "XGBoost (Preprocessed)": ("XGBoost_Preprocessed.joblib", X_train_proc, X_test_proc_df, clean_feature_names),
    "LightGBM (Preprocessed)": ("LightGBM_Preprocessed.joblib", X_train_proc, X_test_proc_df, clean_feature_names),
    "LightGBM (Raw)": ("LightGBM_Raw_Calibrated.joblib", X_train, X_test, X_train.columns.tolist()),
    "CatBoost (Preprocessed)": ("CatBoost_Preprocessed.joblib", X_train_proc, X_test_proc_df, clean_feature_names),
    "CatBoost (Raw)": ("CatBoost_Raw_Calibrated.joblib", X_train, X_test, X_train.columns.tolist()),
    "Random Forest (Preprocessed)": ("Random_Forest_Preprocessed.joblib", X_train_proc, X_test_proc_df, clean_feature_names),
    "Random Forest (Minimal)": ("Random_Forest_Minimal_Calibrated.joblib", X_train_med, pd.DataFrame(X_test_med, columns=X_train.columns), X_train.columns.tolist())
}

cls_ckpt_name, X_tr_cls, X_te_cls_df, cls_feat_names = model_registry[best_classical]
cal_cls_model = joblib.load(os.path.join(MODELS_DIR, cls_ckpt_name))
base_cls_model = cal_cls_model.estimator.__class__(**cal_cls_model.estimator.get_params())
base_cls_model.fit(X_tr_cls, y_train)

def get_tree_shap(model, X_df):
    exp = shap.TreeExplainer(model)
    vals = exp.shap_values(X_df)
    if isinstance(vals, list):
        vals = vals[1] if len(vals) > 1 else vals[0]
    vals = np.array(vals)
    if vals.ndim == 3:
        vals = vals[:, :, 1]
    ev = exp.expected_value
    base_val = float(np.array(ev).flatten()[1]) if isinstance(ev, (list, np.ndarray)) and len(np.array(ev).flatten()) > 1 else float(np.array(ev).flatten()[0])
    return vals, base_val

shap_cls_values, base_cls_ev = get_tree_shap(base_cls_model, X_te_cls_df)

# SHAP Beeswarms for Leading Classical Model (Top 15 and Full)
for max_d, tag in [(15, "Top15"), (len(cls_feat_names), "Full")]:
    plt.figure(figsize=(8 if max_d==15 else 9, 7 if max_d==15 else 11), dpi=300)
    shap.summary_plot(shap_cls_values, X_te_cls_df, max_display=max_d, show=False)
    plt.title(f"{best_classical}: {'Top 15' if max_d==15 else 'Global'} Feature Attribution", fontsize=12, weight="bold", pad=12)
    plt.tight_layout()
    plt.savefig(os.path.join(TARGET_DIR, f"Figure_S3_SHAP_Summary_Classical_{tag}.png"), bbox_inches="tight"); plt.close()

# SHAP Beeswarms for TabPFN (preserving raw NaNs as grey dots)
shap_tabpfn_values = np.load(os.path.join(BASE_DIR, "shap_tabpfn_full.npy"))
if isinstance(shap_tabpfn_values, list):
    shap_tabpfn_values = shap_tabpfn_values[1] if len(shap_tabpfn_values) > 1 else shap_tabpfn_values[0]
shap_tabpfn_values = np.array(shap_tabpfn_values)
if shap_tabpfn_values.ndim == 3:
    shap_tabpfn_values = shap_tabpfn_values[:, :, 1]
base_tab_ev = float(np.load(os.path.join(BASE_DIR, "shap_tabpfn_expected_value.npy"))[0])

for max_d, out_f in [(15, "Figure_SHAP_Summary_TabPFN.png"), (len(X_test.columns), "Figure_SHAP_Summary_TabPFN_Full.png")]:
    plt.figure(figsize=(8 if max_d==15 else 9, 7 if max_d==15 else 11), dpi=300)
    shap.summary_plot(shap_tabpfn_values, X_test, max_display=max_d, show=False)
    plt.title(f"{best_foundation}: {'Top 15' if max_d==15 else 'Global'} Feature Attribution", fontsize=12, weight="bold", pad=12)
    plt.tight_layout()
    plt.savefig(os.path.join(TARGET_DIR, out_f), bbox_inches="tight"); plt.close()

# Export SHAP importance rankings
s_cls_imp = pd.Series(np.mean(np.abs(shap_cls_values), axis=0), index=cls_feat_names, name=f"Mean_Abs_SHAP_{best_classical}")
s_tab_imp = pd.Series(np.mean(np.abs(shap_tabpfn_values), axis=0), index=X_train.columns.tolist(), name="Mean_Abs_SHAP_TabPFN")
df_imp = pd.concat([s_cls_imp, s_tab_imp], axis=1).reset_index().rename(columns={"index": "Feature_Name"})
df_imp = df_imp.sort_values(by="Mean_Abs_SHAP_TabPFN", ascending=False)
df_imp["Rank_TabPFN"] = range(1, len(df_imp) + 1)
df_imp = df_imp.sort_values(by=f"Mean_Abs_SHAP_{best_classical}", ascending=False)
df_imp["Rank_Classical"] = range(1, len(df_imp) + 1)
df_imp.to_excel(os.path.join(TARGET_DIR, "Table_SHAP_Feature_Importance_Rankings.xlsx"), index=False)

# Representative Waterfall explanations (Median Probability & Centroid)
mort_idx, surv_idx = np.where(y_test.values == 1)[0], np.where(y_test.values == 0)[0]
cent_mort = np.median(X_test_proc[mort_idx], axis=0, keepdims=True)
cent_surv = np.median(X_test_proc[surv_idx], axis=0, keepdims=True)

def render_waterfalls(model_label, probs, shap_mat, base_val, data_df, feat_names, prefix):
    idx_m_med = mort_idx[np.argsort(np.abs(probs[mort_idx] - np.median(probs[mort_idx])))[0]]
    idx_s_med = surv_idx[np.argsort(np.abs(probs[surv_idx] - np.median(probs[surv_idx])))[0]]
    idx_m_cen = mort_idx[np.argsort(cdist(X_test_proc[mort_idx], cent_mort).flatten())[0]]
    idx_s_cen = surv_idx[np.argsort(cdist(X_test_proc[surv_idx], cent_surv).flatten())[0]]

    for idx_sel, lbl, suffix in [
        (idx_m_med, "Typical Non-Survivor (Median Prob)", "Mortality_Median"),
        (idx_s_med, "Typical Survivor (Median Prob)", "Survivor_Median"),
        (idx_m_cen, "Typical Non-Survivor (Centroid)", "Mortality_Centroid"),
        (idx_s_cen, "Typical Survivor (Centroid)", "Survivor_Centroid")
    ]:
        exp = shap.Explanation(values=shap_mat[idx_sel], base_values=base_val, data=data_df.iloc[idx_sel].round(2).values, feature_names=feat_names)
        plt.figure(figsize=(8.5, 6), dpi=300)
        shap.plots.waterfall(exp, max_display=10, show=False)
        plt.title(f"{model_label}: {lbl}", fontsize=11, weight="bold", pad=12)
        plt.tight_layout()
        plt.savefig(os.path.join(TARGET_DIR, f"Figure_Waterfall_{prefix}_{suffix}.png"), bbox_inches="tight"); plt.close()

render_waterfalls(best_foundation, predictions_prob[best_foundation], shap_tabpfn_values, base_tab_ev, X_test, X_train.columns.tolist(), "TabPFN")
render_waterfalls(best_classical, predictions_prob[best_classical], shap_cls_values, base_cls_ev, X_te_cls_df, cls_feat_names, "Classical")

# Fix TabPFN Survivor & Mortality Waterfall plots to match Figure S5 exactly
p_tab = predictions_prob["TabPFN 3.5 (Raw)"]
p_shap_implied = base_tab_ev + shap_tabpfn_values.sum(axis=1)
valid_mort = [i for i in mort_idx if p_shap_implied[i] > base_tab_ev + 0.08 and abs(p_shap_implied[i] - p_tab[i]) < 0.05]
valid_surv = [i for i in surv_idx if p_shap_implied[i] < base_tab_ev - 0.08 and abs(p_shap_implied[i] - p_tab[i]) < 0.05]
idx_m_clean = min(valid_mort, key=lambda i: abs(p_tab[i] - np.median(p_tab[mort_idx])))
idx_s_clean = min(valid_surv, key=lambda i: abs(p_tab[i] - np.median(p_tab[surv_idx])))

for idx_sel, lbl, suffix in [
    (idx_m_clean, "Typical Non-Survivor (Median Prob)", "Mortality_Median"),
    (idx_s_clean, "Typical Survivor (Median Prob)", "Survivor_Median")
]:
    exp = shap.Explanation(
        values=shap_tabpfn_values[idx_sel], base_values=base_tab_ev,
        data=X_test.iloc[idx_sel].round(2).values, feature_names=X_train.columns.tolist()
    )
    plt.figure(figsize=(8.5, 6), dpi=300)
    shap.plots.waterfall(exp, max_display=10, show=False)
    plt.title(f"TabPFN 3.5 (Raw): {lbl}", fontsize=11, weight="bold", pad=12)
    plt.tight_layout()
    plt.savefig(os.path.join(TARGET_DIR, f"Figure_Waterfall_TabPFN_{suffix}.png"), bbox_inches="tight")
    plt.close()

# Unified demographic LOWESS dependence (Figure S4)
age_vals, sex_vals = X_test['age_years'].values, X_test['sex_female'].values
shap_age_tab = shap_tabpfn_values[:, X_train.columns.tolist().index('age_years')]
shap_age_cls = shap_cls_values[:, cls_feat_names.index('age_years')]

fig, (ax_tab, ax_cls) = plt.subplots(1, 2, figsize=(15, 5.5), dpi=300)
mask_m, mask_f = (sex_vals == 0), (sex_vals == 1)

for ax, s_age, title_str in [(ax_tab, shap_age_tab, f"(a) {best_foundation}"), (ax_cls, shap_age_cls, f"(b) {best_classical}")]:
    valid_a = ~np.isnan(age_vals)
    ax.scatter(age_vals[mask_m & valid_a], s_age[mask_m & valid_a], color='#4A708B', alpha=0.45, s=32, label='Male')
    ax.scatter(age_vals[mask_f & valid_a], s_age[mask_f & valid_a], color='#D9534F', alpha=0.65, s=36, label='Female')
    lo_ov = lowess(s_age[valid_a], age_vals[valid_a], frac=0.40)
    lo_fe = lowess(s_age[mask_f & valid_a], age_vals[mask_f & valid_a], frac=0.45)
    lo_ma = lowess(s_age[mask_m & valid_a], age_vals[mask_m & valid_a], frac=0.45)
    ax.plot(lo_ov[:, 0], lo_ov[:, 1], color='#1C2D42', linewidth=2.5, label='Trend: Overall')
    ax.plot(lo_fe[:, 0], lo_fe[:, 1], color='#C0392B', linewidth=1.8, linestyle='--', label='Trend: Female')
    ax.plot(lo_ma[:, 0], lo_ma[:, 1], color='#2C3E50', linewidth=1.8, linestyle=':', label='Trend: Male')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(title_str, fontsize=12, weight="bold")
    ax.set_xlabel("Patient Age (years)", fontsize=10.5)
    ax.set_xlim(16, 94); ax.grid(True, linestyle=":", alpha=0.5); ax.legend(frameon=True, facecolor="white", loc="upper left")

ax_tab.set_ylabel("Mortality Attribution (SHAP value for Age)", fontsize=10.5)
max_abs_cls = np.max(np.abs(shap_age_cls)) * 1.15
ax_cls.set_ylim(-max_abs_cls, max_abs_cls)
plt.tight_layout()
plt.savefig(os.path.join(TARGET_DIR, "Figure_S4_Demographic_LOWESS_Official.png"), bbox_inches="tight"); plt.close()

# Bivariate physiological dependence profiles
bivariate_pairs = [
    ('arterial_ph_d0', 'serum_bicarbonate_d0', 'Arterial pH (D0)', 'Serum Bicarbonate (D0)'),
    ('predialysis_creatinine_d_minus_1', 'dialysis_day_creatinine', 'Pre-dialysis Creatinine (D-1)', 'Dialysis Day Creatinine (D0)'),
    ('urea_to_baseline_creatinine_ratio', 'arterial_ph_d0', 'Urea-to-Baseline Creatinine Ratio', 'Arterial pH (D0)')
]

for feat, int_feat, f_lbl, i_lbl in bivariate_pairs:
    if feat in cls_feat_names and int_feat in cls_feat_names:
        valid_biv = X_te_cls_df[[feat, int_feat]].notna().all(axis=1).values
        f_idx, i_idx = cls_feat_names.index(feat), cls_feat_names.index(int_feat)
        plt.figure(figsize=(7, 5), dpi=300)
        shap.dependence_plot(
            f_idx, shap_cls_values[valid_biv], X_te_cls_df.values[valid_biv],
            feature_names=cls_feat_names, interaction_index=i_idx, show=False
        )
        plt.title(f"{best_classical}: {f_lbl} vs {i_lbl}", fontsize=11, weight="bold", pad=12)
        plt.ylabel(f"SHAP value for {f_lbl}"); plt.tight_layout()
        plt.savefig(os.path.join(TARGET_DIR, f"Figure_SHAP_Dependence_{feat}_x_{int_feat}.png"), bbox_inches="tight"); plt.close()

print(f"\nAll evaluation artifacts successfully exported to: {TARGET_DIR}")